## Feature importance for discriminator predictions for CIFAR-10

This notebook can be used to replicate the results reported in  "Generative adversarial learning can explain why imagination seems less real as we grow up" by Ozsu, Petrova, Dekker & Dijkstra.

Scripts would pass 10000 generated imagess through AlexNet and the discriminator from a given epoch. It uses layer-wise activations to predict discriminator predictions using Ridge regression and track the model performance through r-squared. The results of these analyses were reported in Figure 6, Panel A.

To run this scripts, you would need to have the weights available which can be gathered from the OSF files. In this implementation, the weights are imported via Google Drive but other methods are also possible.

For any questions or if you detect a bug: a.ozsu@ucl.ac.uk


In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import os
import pickle
import cv2
import warnings
from timeit import default_timer as timer

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """
    Central configuration for all hyperparameters.
    This class holds all the tunable parameters for the GAN, making it easy to
    see and modify the experimental setup from one place.
    """

    # --- 1. Dataset and Data Paths ---
    dataset_name = "cifar10"  # Options: 'cifar10', 'mnist'
    data_path = "./data"

    # --- 2. Model Architecture ---
    latent_dim = 128
    generator_feature_maps = 128
    critic_feature_maps = 128
    image_channels = 3

    # --- 3. WGAN-GP Training Parameters ---
    batch_size = 64
    num_epochs = 100
    learning_rate_generator = 0.0001
    learning_rate_critic = 0.0003
    adam_beta1 = 0.0
    adam_beta2 = 0.9
    critic_updates_per_generator_update = 1

    # --- 4. Stability and Regularization ---
    gradient_penalty_weight = 10


    # --- 5. Monitoring, Logging, and Saving ---
    monitor_every_steps = 50
    save_samples_every_steps = 200
    save_model_every_epochs = 5
    save_activations_every_epochs = 1  # Set to 1 to save activations every epoch

    # --- 6. File Paths ---
    local_path = "/content/GAN_Outputs_Final"
    output_dir = os.path.join(local_path, "outputs")
    checkpoint_dir = os.path.join(local_path, "checkpoints")
    activation_dir = os.path.join(local_path, "activations")

    def __init__(self):
        """Creates all necessary output directories to prevent FileNotFoundError."""
        if self.dataset_name == "mnist":
            self.image_channels = 1

        self.samples_dir = os.path.join(self.output_dir, "samples")
        self.monitoring_dir = os.path.join(self.output_dir, "monitoring")
        self.grad_cam_dir = os.path.join(self.output_dir, "grad_cam")

        os.makedirs(self.samples_dir, exist_ok=True)
        os.makedirs(self.monitoring_dir, exist_ok=True)
        os.makedirs(self.grad_cam_dir, exist_ok=True)
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.activation_dir, exist_ok=True)


config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Outputs will be saved to: {config.local_path}")

Using device: cuda
Outputs will be saved to: /content/GAN_Outputs_Final


In [ ]:

# ============================================================================
# MODELS (GENERATOR & CRITIC)
# ============================================================================
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.activations = {}
        self.main = nn.Sequential(
            nn.ConvTranspose2d(
                config.latent_dim,
                config.generator_feature_maps * 8,
                4,
                1,
                0,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 8,
                config.generator_feature_maps * 4,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 4,
                config.generator_feature_maps * 2,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 2,
                config.generator_feature_maps,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps,
                config.image_channels,
                3,
                1,
                1,
                bias=False,
            ),
            nn.Tanh(),
        )
        self._initialize_weights()
        self._register_hooks()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.orthogonal_(m.weight, gain=0.8)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def _register_hooks(self):
        def get_hook(name):
            def hook(model, input, output):
                self.activations[name] = output.detach()


            return hook

        for i, layer in enumerate(self.main):
            if isinstance(layer, nn.ReLU) or isinstance(layer, nn.Tanh):
                layer.register_forward_hook(get_hook(f"gen_layer_{i}"))

    def forward(self, z):
        return self.main(z)

    def clear_activations(self):
        self.activations.clear()

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.activations = {}
        self.gradients = {}
        self.layer1 = nn.Sequential(
            nn.Conv2d(
                config.image_channels, config.critic_feature_maps, 4, 2, 1, bias=False
            ),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(
                config.critic_feature_maps,
                config.critic_feature_maps * 2,
                4,
                2,
                1,
                bias=False,
            ),
            nn.InstanceNorm2d(config.critic_feature_maps * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(
                config.critic_feature_maps * 2,
                config.critic_feature_maps * 4,
                4,
                2,
                1,
                bias=False,
            ),
            nn.InstanceNorm2d(config.critic_feature_maps * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=False),
        )
        self.final_conv = nn.Conv2d(
            config.critic_feature_maps * 4, 1, 4, 1, 0, bias=False
        )
        self._initialize_weights()
        self._register_hooks()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.orthogonal_(m.weight, gain=0.8)
            elif isinstance(m, nn.InstanceNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def _register_hooks(self):
        def get_activation_hook(name):
            def hook(model, input, output):
                self.activations[name] = output

            return hook

        def get_gradient_hook(name):
            def hook(model, grad_in, grad_out):
                self.gradients[name] = grad_out[0]

            return hook

        self.layer1[0].register_forward_hook(get_activation_hook("layer1"))
        self.layer1[0].register_full_backward_hook(get_gradient_hook("layer1"))
        self.layer2[0].register_forward_hook(get_activation_hook("layer2"))
        self.layer2[0].register_full_backward_hook(get_gradient_hook("layer2"))
        self.layer3[0].register_forward_hook(get_activation_hook("layer3"))
        self.layer3[0].register_full_backward_hook(get_gradient_hook("layer3"))

    def forward(self, x):
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        h3 = self.layer3(h2)
        out = self.final_conv(h3)
        return out.view(-1)

    def clear_hooks_data(self):
        self.activations.clear()
        self.gradients.clear()

In [ ]:
!unzip "/content/drive/MyDrive/checkpoints_newarch.zip" -d "/content"

Archive:  /content/drive/MyDrive/checkpoints_newarch.zip
   creating: /content/checkpoints/
  inflating: /content/checkpoints/C_epoch_1.pth  
  inflating: /content/checkpoints/C_epoch_10.pth  
  inflating: /content/checkpoints/C_epoch_100.pth  
  inflating: /content/checkpoints/C_epoch_11.pth  
  inflating: /content/checkpoints/C_epoch_12.pth  
  inflating: /content/checkpoints/C_epoch_13.pth  
  inflating: /content/checkpoints/C_epoch_14.pth  
  inflating: /content/checkpoints/C_epoch_15.pth  
  inflating: /content/checkpoints/C_epoch_16.pth  
  inflating: /content/checkpoints/C_epoch_17.pth  
  inflating: /content/checkpoints/C_epoch_18.pth  
  inflating: /content/checkpoints/C_epoch_19.pth  
  inflating: /content/checkpoints/C_epoch_2.pth  
  inflating: /content/checkpoints/C_epoch_20.pth  
  inflating: /content/checkpoints/C_epoch_21.pth  
  inflating: /content/checkpoints/C_epoch_22.pth  
  inflating: /content/checkpoints/C_epoch_23.pth  
  inflating: /content/checkpoints/C_epoch_

In [ ]:
# ============================================================================
# DATASET LOADING
# ============================================================================
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import os

# ============================================================================
# DATASET LOADING
# ============================================================================
def get_dataloader():
    transform = transforms.Compose(
        [
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5,) * config.image_channels, (0.5,) * config.image_channels
            ),
        ]
    )
    dataset_class = (
        torchvision.datasets.CIFAR10
        if config.dataset_name == "cifar10"
        else torchvision.datasets.MNIST
    )
    dataset = dataset_class(
        root=config.data_path, train=True, transform=transform, download=True
    )
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
    )



# ============================================================================
# IMPORTS
# ============================================================================
import torch
import torch.nn as nn
from torchvision import models
import torchvision
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd
import torch.nn.functional as F
import gc
from google.colab import files


# ============================================================================
# SETTINGS
# ============================================================================
START_EPOCH = 0
END_EPOCH = 100

numImgs = 10000          # 5000 fake + 5000 real = 10000 total samples
batch_size = 128
out_dir = "./mse_results"
os.makedirs(out_dir, exist_ok=True)
checkpoint_dir = "/content/checkpoints"


# ============================================================================
# ALEXNET SETUP
# ============================================================================
alexnet = models.alexnet(pretrained=True).to(device).eval()

layer_names = ['conv1', 'conv2', 'conv3', 'conv4', 'conv5', 'fc1', 'fc2', 'fc3']
layers = {
    'conv1': alexnet.features[0],
    'conv2': alexnet.features[3],
    'conv3': alexnet.features[6],
    'conv4': alexnet.features[8],
    'conv5': alexnet.features[10],
    'fc1':   alexnet.classifier[1],
    'fc2':   alexnet.classifier[4],
    'fc3':   alexnet.classifier[6],
}

activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

for name, layer in layers.items():
    layer.register_forward_hook(get_activation(name))


# ============================================================================
# DATA + STATIC NOISE
# ============================================================================
real_loader = get_dataloader()
static_noise = torch.randn(numImgs, config.latent_dim, 1, 1, device=device)


# ============================================================================
# HELPERS
# ============================================================================
def stack_feature_batches(batch_list):
    if len(batch_list) == 0:
        return np.empty((0, 0))
    return np.concatenate(batch_list, axis=0)


def preprocess_for_alexnet(imgs):
    """
    Convert GAN/dataset images from [-1, 1] to AlexNet ImageNet format.
    """
    imgs = (imgs + 1) / 2
    imgs = imgs.clamp(0, 1)
    imgs = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)

    mean = torch.tensor([0.485, 0.456, 0.406], device=imgs.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=imgs.device).view(1, 3, 1, 1)
    imgs = (imgs - mean) / std
    return imgs


def process_image_batch(imgs, C, alexnet, all_feats, scores_all, labels_all, label_name):
    """
    Process one batch of images through Critic + AlexNet.
    Stores:
      - critic score
      - AlexNet layer features
      - sample label ('fake' or 'real')
    """
    with torch.no_grad():
        # Critic scores
        s = C(imgs).detach().cpu().numpy()
        if s.ndim > 1:
            s = s.reshape(s.shape[0])
        scores_all.append(s)

        # Save labels
        labels_all.extend([label_name] * imgs.size(0))

        # AlexNet features
        alex_imgs = preprocess_for_alexnet(imgs)
        activations.clear()
        _ = alexnet(alex_imgs)

        for l in layer_names:
            f = activations[l]
            if f.dim() == 4:
                f = f.mean(dim=[2, 3])   # spatial average for conv layers
            all_feats[l].append(f.detach().cpu().numpy())


def generate_and_collect(G, C, noise_tensor, real_loader, numImgs=numImgs, batch_size=batch_size):
    """
    Collect:
      - numImgs fake images
      - numImgs real images

    Returns:
        all_feats: dict[layer] -> [2*numImgs, C]
        scores_all: [2*numImgs]
        labels_all: [2*numImgs] array of 'fake'/'real'
    """
    all_feats = {l: [] for l in layer_names}
    scores_all = []
    labels_all = []

    # ------------------ FAKE IMAGES ------------------
    fake_count = 0
    with torch.no_grad():
        for i in range(0, min(noise_tensor.shape[0], numImgs), batch_size):
            z = noise_tensor[i:i+batch_size].to(device)
            fake_imgs = G(z)

            process_image_batch(
                fake_imgs, C, alexnet,
                all_feats, scores_all, labels_all,
                label_name="fake"
            )

            fake_count += fake_imgs.size(0)
            if fake_count >= numImgs:
                break

    # ------------------ REAL IMAGES ------------------
    #real_count = 0
    #with torch.no_grad():
     #   for real_imgs, _ in real_loader:
      #      real_imgs = real_imgs.to(device)

       #     remaining = numImgs - real_count
        #    if remaining <= 0:
         #       break

          #  if real_imgs.size(0) > remaining:
           #     real_imgs = real_imgs[:remaining]

            #process_image_batch(
             #   real_imgs, C, alexnet,
              #  all_feats, scores_all, labels_all,
               # label_name="real"
            #)

            #real_count += real_imgs.size(0)
            #if real_count >= numImgs:
             #   break

    # Stack into arrays
    for l in layer_names:
        all_feats[l] = stack_feature_batches(all_feats[l])

    scores_all = np.concatenate(scores_all, axis=0)
    labels_all = np.array(labels_all)

    print(f"Collected {np.sum(labels_all == 'fake')} fake images")
    print(f"Collected {np.sum(labels_all == 'real')} real images")
    print(f"Total samples: {len(labels_all)}")

    return all_feats, scores_all, labels_all

from sklearn.metrics import r2_score

def ridge_cv_mse_with_permutation(
    X, y,
    alpha=1.0, n_splits=5, n_perm=1000, random_state=42
):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    y_pred_all = np.zeros_like(y, dtype=float)
    y_true_all = np.zeros_like(y, dtype=float)

    nrmse_scores = []
    r2_scores = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        y_train_mean = y_train.mean()
        y_train_std = y_train.std()

        if y_train_std < 1e-12:
            y_train_std = 1.0

        y_train_z = (y_train - y_train_mean) / y_train_std
        y_test_z  = (y_test  - y_train_mean) / y_train_std

        model = Ridge(alpha=alpha, fit_intercept=True)
        model.fit(X_train, y_train_z)

        y_pred_z = model.predict(X_test)

        r2_scores.append(r2_score(y_test_z, y_pred_z))
        nrmse = np.sqrt(mean_squared_error(y_test_z, y_pred_z))
        nrmse_scores.append(nrmse)

        y_pred_all[test_idx] = y_pred_z
        y_true_all[test_idx] = y_test_z

    actual_rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
    actual_r2 = r2_score(y_true_all, y_pred_all)

    rng = np.random.default_rng(random_state)

    # ------------------------------------------------------------------
    # combinded permutation test
    # ------------------------------------------------------------------
    perm_rmse = np.zeros(n_perm)
    perm_r2 = np.zeros(n_perm)

    for p in range(n_perm):
        y_perm = rng.permutation(y_true_all)
        perm_rmse[p] = np.sqrt(mean_squared_error(y_perm, y_pred_all))
        perm_r2[p] = r2_score(y_perm, y_pred_all)

    p_value = (np.sum(perm_rmse <= actual_rmse) + 1) / (n_perm + 1)
    p_value_r2 = (np.sum(perm_r2 >= actual_r2) + 1) / (n_perm + 1)

    result = {
        "cv_nrmse_mean": float(np.mean(nrmse_scores)),
        "cv_nrmse_std":  float(np.std(nrmse_scores)),
        "actual_rmse":   float(actual_rmse),
        "perm_rmse_mean": float(np.mean(perm_rmse)),
        "perm_rmse_std":  float(np.std(perm_rmse)),
        "p_value":        float(p_value),
        "n_perm":         n_perm,
        "alpha":          alpha,
        "cv_r2_mean":     float(np.mean(r2_scores)),
        "cv_r2_std":      float(np.std(r2_scores)),
        "actual_r2":      float(actual_r2),
        "perm_r2_mean":   float(np.mean(perm_r2)),
        "perm_r2_std":    float(np.std(perm_r2)),
        "p_value_r2":     float(p_value_r2),
    }

    return result, perm_r2

from statsmodels.stats.multitest import multipletests
# ============================================================================
# MAIN LOOP
# ============================================================================
all_results        = []
all_contrast_rows  = []

for epoch in range(START_EPOCH, END_EPOCH + 1):
    print("\n==============================")
    print(f"Epoch {epoch}: Loading Generator and Critic")
    print("==============================")

    # --- Load Generator ---
    try:
        G_eval = Generator().to(device)
        if epoch > 0:
            g_path = os.path.join(checkpoint_dir, f"G_epoch_{epoch}.pth")
            G_eval.load_state_dict(torch.load(g_path, map_location=device))
        G_eval.eval()
    except FileNotFoundError:
        print(f"WARNING: Generator checkpoint for epoch {epoch} not found. Skipping.")
        continue

    # --- Load Critic ---
    try:
        C_eval = Critic().to(device)
        if epoch > 0:
            c_path = os.path.join(checkpoint_dir, f"C_epoch_{epoch}.pth")
            C_eval.load_state_dict(torch.load(c_path, map_location=device))
        C_eval.eval()
    except FileNotFoundError:
        print(f"WARNING: Critic checkpoint for epoch {epoch} not found. Skipping.")
        continue

    noise_to_use = static_noise[:numImgs]

    all_feats, scores, labels = generate_and_collect(
        G_eval, C_eval, noise_to_use,
        real_loader=real_loader, numImgs=numImgs, batch_size=batch_size
    )

    # ── per-epoch storage ───────────────────────────────────────────────────
    epoch_results  = {}   # layer -> summary dict
    epoch_perm_r2  = {}   # layer -> perm_r2 array  (length = n_perm)

    N_PERM = 1000

    for l in layer_names:
        X = all_feats[l]
        y = scores.copy()

        res, perm_r2_arr = ridge_cv_mse_with_permutation(
            X, y,
            alpha=1.0, n_splits=5, n_perm=N_PERM,
            random_state=42 + epoch
        )
        res.update({"layer": l, "epoch": epoch})
        all_results.append(res)

        epoch_results[l] = res
        epoch_perm_r2[l] = perm_r2_arr

    conv3_actual  = epoch_results["conv3"]["actual_r2"]
    conv3_perm    = epoch_perm_r2["conv3"]

    other_layers  = [l for l in layer_names if l != "conv3"]
    raw_p_values  = []
    contrast_meta = []

    for l in other_layers:
        obs_diff  = conv3_actual - epoch_results[l]["actual_r2"]
        null_diff = conv3_perm   - epoch_perm_r2[l]

        p_raw = (np.sum(null_diff >= obs_diff) + 1) / (N_PERM + 1)

        raw_p_values.append(p_raw)
        contrast_meta.append({
            "epoch":          epoch,
            "layer_x":        l,
            "conv3_r2":       conv3_actual,
            "layer_x_r2":     epoch_results[l]["actual_r2"],
            "obs_diff_r2":    obs_diff,
            "null_diff_mean": float(null_diff.mean()),
            "null_diff_std":  float(null_diff.std()),
            "p_raw":          p_raw,
        })

    # FDR correction
    reject, p_fdr, _, _ = multipletests(raw_p_values, alpha=0.05, method="fdr_bh")

    for i, meta in enumerate(contrast_meta):
        meta["p_fdr"]  = float(p_fdr[i])
        meta["reject"] = bool(reject[i])
        all_contrast_rows.append(meta)

    print(f"\n── Epoch {epoch}: conv3 vs others (FDR-corrected) ──")
    for meta in contrast_meta:
        sig = "✓ REJECT" if meta["reject"] else "  n.s."
        print(f"  conv3 vs {meta['layer_x']:5s} | "
              f"obs Δr²={meta['obs_diff_r2']:+.4f} | "
              f"p_raw={meta['p_raw']:.4f} | "
              f"p_fdr={meta['p_fdr']:.4f}  {sig}")

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================================
# SAVE RESULTS
# ============================================================================
df_mse      = pd.DataFrame(all_results)
df_contrast = pd.DataFrame(all_contrast_rows)

csv_path          = os.path.join(out_dir, "MSE_CV_over_epochs_CIFAR.csv")
csv_contrast_path = os.path.join(out_dir, "conv3_contrast_over_epochs_CIFAR.csv")

df_mse.to_csv(csv_path, index=False)
df_contrast.to_csv(csv_contrast_path, index=False)

print("✅ Saved MSE results to:", csv_path)
print("✅ Saved conv3 contrast results to:", csv_contrast_path)

files.download(csv_path)
files.download(csv_contrast_path)


Epoch 0: Loading Generator and Critic
Collected 10000 fake images
Collected 0 real images
Total samples: 10000

── Epoch 0: conv3 vs others (FDR-corrected) ──
  conv3 vs conv1 | obs Δr²=+0.0190 | p_raw=0.0010 | p_fdr=0.0017  ✓ REJECT
  conv3 vs conv2 | obs Δr²=+0.0079 | p_raw=0.0010 | p_fdr=0.0017  ✓ REJECT
  conv3 vs conv4 | obs Δr²=+0.0055 | p_raw=0.0010 | p_fdr=0.0017  ✓ REJECT
  conv3 vs conv5 | obs Δr²=+0.0129 | p_raw=0.0010 | p_fdr=0.0017  ✓ REJECT
  conv3 vs fc1   | obs Δr²=-0.1069 | p_raw=1.0000 | p_fdr=1.0000    n.s.
  conv3 vs fc2   | obs Δr²=-0.0332 | p_raw=1.0000 | p_fdr=1.0000    n.s.
  conv3 vs fc3   | obs Δr²=-0.0162 | p_raw=1.0000 | p_fdr=1.0000    n.s.

Epoch 1: Loading Generator and Critic
Collected 10000 fake images
Collected 0 real images
Total samples: 10000

── Epoch 1: conv3 vs others (FDR-corrected) ──
  conv3 vs conv1 | obs Δr²=+0.1798 | p_raw=0.0010 | p_fdr=0.0014  ✓ REJECT
  conv3 vs conv2 | obs Δr²=+0.0772 | p_raw=0.0010 | p_fdr=0.0014  ✓ REJECT
  conv3 vs 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>